# Logits Alignments

In [1]:
def default_params(): 
    return {
        'current_model': 'M1',
        'quantization': 'none', #['none',"int4", "int8", "float32", "float16"]
        'dataset': {
            'path': '/workspaces/CodeSmells/semeru-datasets/code_smells/transformation',
            'transformation': 'SwitchRelation',
            'content_column': 'code',
            'sampling_size': 500,
        },
        
        'logging_path': '/workspaces/CodeSmells/datax/code_smells/logs', 
        'raw_logits_path' : '/workspaces/CodeSmells/datax/code_smells/logits/transformation',
        'alignments_path': '/workspaces/CodeSmells/data/extension/transformation/alignments',
        'cache_dir': '/workspaces/CodeSmells/datax/hugging_face_cache',
      
        'causal_models': {
            'M1' : 'codellama/CodeLlama-7b-hf', #https://huggingface.co/codellama/CodeLlama-7b-hf, 
            'M2' : 'mistralai/Mistral-7B-v0.3', #https://huggingface.co/mistralai/Mistral-7B-v0.3,
            'M3' : 'microsoft/Phi-3.5-mini-instruct', #https://huggingface.co/microsoft/Phi-3.5-mini-instruct 
            'M4' : 'Qwen/Qwen2.5-Coder-7B', #https://huggingface.co/Qwen/Qwen2.5-Coder-7B
            'M5' : 'facebook/incoder-6B', #https://huggingface.co/facebook/incoder-6B
            'M6' : 'bigcode/starcoder2-7b', #https://huggingface.co/bigcode/starcoder2-7b 
            'M7' : 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B', #https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Llama-8B
            'M8' : 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', #https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
        },
    }
params = default_params()


#### Imports

In [2]:
import pandas as pd
import random
import numpy as np
from statistics import mean, median
import os
import torch
import gc
from difflib import SequenceMatcher
from scipy.stats import entropy

In [3]:
from datasets import load_dataset, Dataset

In [4]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

2025-03-27 19:14:43.446556: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743102883.463932 1952124 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743102883.469268 1952124 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-27 19:14:43.486517: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [5]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [6]:
# Define log file path
log_file = f"{params['logging_path']}/{params['current_model']}/{params['dataset']['transformation']}"
create_folder(log_file)
log_file += '/align_aggr.txt'

# Create the log file if it doesn't exist
if not os.path.exists(log_file):
    with open(log_file, 'w'): 
        pass  # Create an empty log file

In [7]:
import logging
logging.basicConfig(filename=log_file, format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

#### Model Loading

In [8]:
def instantiate_llm(model_name:str, cache_dir:str):
     '''Instantiate AutoModelForCausalLM'''
     tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoTokenizer - " + model_name)
     model = None
     match params['quantization']:
               case 'int4':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_4bit=True)
               case 'int8':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_8bit=True)
               case 'float32':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float32)
               case 'float16':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float16)
               case _: 
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoModelForCausalLM - " + model_name)

     return tokenizer, model

In [9]:
tokenizer, model = instantiate_llm(params['causal_models'][params['current_model']], params['cache_dir'])

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

#### Load Dataset

In [10]:
df_actual_ntp = pd.read_json(f"{params['raw_logits_path']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['transformation']}/raw_logits.json")

In [11]:
df_actual_ntp.head(2)

,id,commit_id,repo,path,file_name,commit_message,url,language,category,code,...,ast_levels,n_ast_nodes,n_ast_errors,n_identifiers,input_ids,input_lenght,max_prob,min_prob,actual_prob,loss
0,256261,a59bca366174d9c692fa19750c24d65f47660ef7,haystack,haystack/modeling/training/base.py,base.py,Apply black formatting (#2115)\n\n* Testing bl...,https://github.com/deepset-ai/haystack.git,Python,Warning,def _get_state_dict(self):\n \n ...,...,9,206,0,20,"[822, 903, 657, 29918, 3859, 29918, 8977, 2989...",295,"[[<PRE>, 0.7492886186000001], [module, 0.47944...","[[<s>, 0.0], [$}, 1e-10], [oreferrer, 0.0], [o...","[[def, 0.0007611339000000001], [_, 0.009010307...",0.650187
1,305801,6f564e4f514b56bce281ec7e82703cfbff87b417,core,homeassistant/components/ring/binary_sensor.py,binary_sensor.py,Improve entity type hints [r] (#77874),https://github.com/home-assistant/core.git,Python,Warning,async def async_added_to_hass(self) -> None:\n...,...,10,60,0,6,"[7465, 822, 7465, 29918, 23959, 29918, 517, 29...",72,"[[<PRE>, 0.7492926717], [\n, 0.145589932800000...","[[<s>, 0.0], [oreferrer, 4e-10], [$}, 1.100000...","[[async, 1.08118e-05], [def, 0.0006238665], [a...",1.363934


#### Token Binding

In [12]:
def find_range_of_indexes(positions, search_range):
    """
    Finds the range of indexes in the positions array where the search_range is fully included.
    
    Args:
    - positions: A list of tuples, where each tuple is (start_position, end_position) (inclusive).
    - search_range: A tuple (start_position, end_position), where start_position is inclusive and end_position is exclusive.
    
    Returns:
    - A tuple (start_index, end_index) representing the range of indexes in the positions array where the search_range is included.
    """
    start, end = search_range
    start_index = -1
    end_index = -1

    for i, (pos_start, pos_end) in enumerate(positions):
        if pos_start <= start <= pos_end:  # Find the start of the range
            start_index = i
        if pos_start <= end - 1 <= pos_end and pos_end>=end:  # Find the end of the range
            end_index = i
            break

    if start_index != -1 and end_index != -1:
        return (start_index, end_index)
    else:
        return None  # If no range is found

In [13]:
def get_substring_positions(code: str, code_smell: str, start):
    """
    Calculate the start and end positions of the substring based on line and column information.

    Parameters:
    text (str): The input string containing multiple lines.
    start (tuple): A tuple of (start_line, start_column) indicating the start position.
    end (tuple): A tuple of (end_line, end_column) indicating the end position.

    Returns:
    tuple: A tuple containing (start_position, end_position) of the substring in the input string.
    """
    lines = code.split('\n')  # Split the string into lines

    # Calculate the character position for the start of the substring
    start_line, start_column = start
    
    try:
        start_position = sum(len(lines[i]) + 1 for i in range(start_line - 1)) + start_column
    except:
        start_position = code.find(code_smell)

    if start_line > len(lines) or start_position>= len(code): 
        start_position = code.find(code_smell)

    if start_position == -1:
        match = SequenceMatcher(None, code, code_smell).find_longest_match()
        start_position= match.a
        end_position = match.a + match.size
    else:
        end_position = start_position + len(code_smell)
        end_position = len(code) if end_position >= len(code) else end_position
    

    return (start_position, end_position)

In [14]:
def find_code_smell_logits(code, code_smell_pos, logits_array, tokenizer):
    indexes_range = find_range_of_indexes(tokenizer.encode_plus(code, return_offsets_mapping=True, add_special_tokens=False)['offset_mapping'], code_smell_pos)
    return logits_array[indexes_range[0]:indexes_range[1]+1]

In [15]:
df_actual_ntp.columns

Index(['id', 'commit_id', 'repo', 'path', 'file_name', 'commit_message', 'url',
       'language', 'category', 'code', 's_msg_id', 's_line', 's_column',
       's_end_line', 's_end_column', 's_code', 'n_whitespaces', 'n_words',
       'vocab_size', 'fun_name', 'complexity', 'nloc', 'token_counts',
       'ast_errors', 'ast_levels', 'n_ast_nodes', 'n_ast_errors',
       'n_identifiers', 'input_ids', 'input_lenght', 'max_prob', 'min_prob',
       'actual_prob', 'loss'],
      dtype='object')

In [16]:
df_actual_ntp['code_smell_pos'] = df_actual_ntp.apply(lambda row: get_substring_positions(row['code'], row['s_code'], (row['s_line'], row['s_column'])), axis=1)

In [17]:
df_actual_ntp.columns

Index(['id', 'commit_id', 'repo', 'path', 'file_name', 'commit_message', 'url',
       'language', 'category', 'code', 's_msg_id', 's_line', 's_column',
       's_end_line', 's_end_column', 's_code', 'n_whitespaces', 'n_words',
       'vocab_size', 'fun_name', 'complexity', 'nloc', 'token_counts',
       'ast_errors', 'ast_levels', 'n_ast_nodes', 'n_ast_errors',
       'n_identifiers', 'input_ids', 'input_lenght', 'max_prob', 'min_prob',
       'actual_prob', 'loss', 'code_smell_pos'],
      dtype='object')

#### Aggregation Functions

In [18]:
def compute_relative_psc(actual_probs, min_probs, max_probs, epsilon=1e-9):
    # Normalize probabilities
    relative_probs = (actual_probs - min_probs) / (max_probs - min_probs + epsilon)

    # Compute relative PSC as the mean of relative probabilities
    relative_psc = np.mean(relative_probs)
    
    return relative_psc

In [19]:
def compute_psc_entropy_scaled(actual_probs, min_probs, max_probs, temperature=1.0, epsilon=1e-9):
    # Apply temperature scaling
    scaled_probs = np.exp(actual_probs / temperature) / np.sum(np.exp(actual_probs / temperature))

    # Compute Shannon entropy per token
    entropy_scores = entropy(scaled_probs, base=2)  # Base 2 for information entropy

    # Normalize using entropy-based weighting
    entropy_norm = 1 - (entropy_scores / np.log2(len(actual_probs) + epsilon))  # Normalize between 0 and 1

    # Compute final PSC score (weighted by entropy)
    adjusted_psc = np.mean(entropy_norm * scaled_probs)

    return adjusted_psc

#### Execute

In [20]:
# Alignments
df_actual_ntp['code_smell_actual_logits'] = df_actual_ntp.apply(lambda row: find_code_smell_logits(row['code'], row['code_smell_pos'], row['actual_prob'], tokenizer), axis=1)
df_actual_ntp['code_smell_max_logits'] = df_actual_ntp.apply(lambda row: find_code_smell_logits(row['code'], row['code_smell_pos'], row['max_prob'], tokenizer), axis=1)
df_actual_ntp['code_smell_min_logits'] = df_actual_ntp.apply(lambda row: find_code_smell_logits(row['code'], row['code_smell_pos'], row['min_prob'], tokenizer), axis=1)

In [21]:
## Aggregations - median
df_actual_ntp['code_smell_actual_prob_median'] = df_actual_ntp.apply(lambda row: median([logit_tuple[1] for logit_tuple in row['code_smell_actual_logits']]), axis=1)
df_actual_ntp['code_smell_max_prob_median'] = df_actual_ntp.apply(lambda row: median([logit_tuple[1] for logit_tuple in row['code_smell_max_logits']]), axis=1)
df_actual_ntp['code_smell_min_prob_median'] = df_actual_ntp.apply(lambda row: median([logit_tuple[1] for logit_tuple in row['code_smell_min_logits']]), axis=1)

In [22]:
## Aggregations  - mean
df_actual_ntp['code_smell_actual_prob_mean'] = df_actual_ntp.apply(lambda row: mean([logit_tuple[1] for logit_tuple in row['code_smell_actual_logits']]), axis=1)
df_actual_ntp['code_smell_max_prob_mean'] = df_actual_ntp.apply(lambda row: mean([logit_tuple[1] for logit_tuple in row['code_smell_max_logits']]), axis=1)
df_actual_ntp['code_smell_min_prob_mean'] = df_actual_ntp.apply(lambda row: mean([logit_tuple[1] for logit_tuple in row['code_smell_min_logits']]), axis=1)

In [23]:
## Aggregations - entropy scaled
df_actual_ntp['code_smell_psc_entropy'] = df_actual_ntp.apply(lambda row: compute_psc_entropy_scaled(np.array([logit_tuple[1] for logit_tuple in row['code_smell_actual_logits']]), np.array([logit_tuple[1] for logit_tuple in row['code_smell_min_logits']]), np.array([logit_tuple[1] for logit_tuple in row['code_smell_max_logits']])), axis=1)
## Aggregations - normalized relative probabilities
df_actual_ntp['code_smell_psc_relative'] = df_actual_ntp.apply(lambda row: compute_relative_psc(np.array([logit_tuple[1] for logit_tuple in row['code_smell_actual_logits']]), np.array([logit_tuple[1] for logit_tuple in row['code_smell_min_logits']]), np.array([logit_tuple[1] for logit_tuple in row['code_smell_max_logits']])), axis=1)

In [27]:
df_actual_ntp[['code', 's_code', 's_msg_id', 'code_smell_psc_relative']]

,code,s_code,s_msg_id,code_smell_psc_relative
0,def _get_state_dict(self):\n \n ...,state_dict = {,W0311,0.750890
1,async def async_added_to_hass(self) -> None:\n...,self._dings_update_callback(),W0311,0.497712
2,def test_new_comment(self):\n post_data...,self.assertEqual(mail.outbox[0].subjec...,C0301,0.891551
3,"def __init__(self, **kwargs) -> None:\n ...",import_str = 'stable_baselines3',W0311,0.679737
4,"def state(self, value):\n state_value =...","self._start_date = max(self._state, se...",W0311,0.749832
...,...,...,...,...
19779,"def test_send_message_queue(self, mock_get_con...",mock.call()\n .__enter__()\n ...,C2801,0.869983
19780,"def _print_Permutation(self, expr):\n f...",Cycle(expr)(expr.size - 1).__repr__(),C2801,0.749144
19781,def test__rshift__() -> None:\n data_a = np...,tensor_a.__rshift__(tensor_b),C2801,1.000000
19782,"def set_executor(self, to_mock_model_controlle...",config_patch.__enter__(),C2801,0.942967


#### SAVE

In [25]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [26]:
alignments_dir = f"{params['alignments_path']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['transformation']}"
create_folder(alignments_dir)
df_actual_ntp.to_json(f"{alignments_dir}/aligned_smells.json")

In [27]:
torch.cuda.empty_cache()
gc.collect()

0